In [1]:
# import packages
import xarray as xr
import geopandas as gpd
import rioxarray as rxr
import odc.geo.xr
import pandas as pd
from pathlib import Path
from geocube.api.core import make_geocube
from rasterio.enums import Resampling
from datetime import datetime, timedelta
import numpy as np
import dask
from dateutil.relativedelta import relativedelta

In [2]:
# Define WHO air quality thresholds for various pollutants.
# These thresholds are used to assess compliance with air quality guidelines.
_aq_complience_limits = {
    'SO2_AQG': 40, # Sulfur Dioxide Air Quality Guideline limit
    'SO2_AP': '1D', # Averaging Period for SO2
    'NO2_AQG': 25, # Nitrogen Dioxide Air Quality Guideline limit
    'NO2_AP': '1D', # Averaging Period for NO2
    'CO_AQG': 4,   # Carbon Monoxide Air Quality Guideline limit
    'CO_AP': '1D', # Averaging Period for CO
    'O3_AQG': 100, # Ozone Air Quality Guideline limit
    'O3_AP': '8H', # Averaging Period for O3
    'PM2P5_AQG': 15, # Fine Particulate Matter (PM2.5) Air Quality Guideline limit
    'PM2P5_AP': '1D', # Averaging Period for PM2.5
    'PM10_AQG': 45, # Coarse Particulate Matter (PM10) Air Quality Guideline limit
    'PM10_AP': '1D' # Averaging Period for PM10
}

# Define AQI (Air Quality Index) pollutant concentration breakpoints.
# These breakpoints are used to calculate the AQI for different pollutants.
# Each inner list represents a range: [low_aqi, high_aqi, low_breakpoint, high_breakpoint]
# _aqi_break = {
#     "CO": [[0.0, 50.0, 0.0, 4.4],      # Good
#         [51.0, 100.0, 4.5, 9.4],     # Moderate
#         [101.0, 150.0, 9.5, 12.4],    # Unhealthy for Sensitive Groups
#         [151.0, 200.0, 12.5, 15.4],   # Unhealthy
#         [201.0, 300.0, 15.5, 30.4],   # Very Unhealthy
#         [301.0, 500.0, 30.5, 60.4]],  # Hazardous
#     "SO2": [[0.0, 50.0, 0.0, 35],
#              [51.0, 100.0, 36, 75],
#              [101.0, 150.0, 76, 185],
#              [151.0, 200.0, 186, 304],
#              [201.0, 300.0, 305, 604],
#              [301.0, 500.0, 605, 1200]],
#     "NO2": [[0.0, 50.0, 0.0, 53],
#              [51.0, 100.0, 54, 100],
#              [101.0, 150.0, 101, 360],
#              [151.0, 200.0, 361, 649],
#              [201.0, 300.0, 650, 1249],
#              [301.0, 500.0, 1250, 2500]],
#     "O3": [[0.0, 50.0, 0.0, 0.054],
#            [51.0, 100.0, 0.055,0.070],
#            [101.0, 150.0, 0.071, 0.085],
#            [151.0, 200.0, 0.086, 0.105],
#            [201.0, 300.0, 0.106, 0.200],
#            [301.0, 500.0, 0.201, 0.5]],
#     "PM10":[[0.0, 50.0, 0.0, 54],
#              [51.0, 100.0, 55, 154],        # FIX: Was 54-154
#              [101.0, 150.0, 155, 254],
#              [151.0, 200.0, 255, 354],
#              [201.0, 300.0, 355, 424],
#              [301.0, 500.0, 425, 604]],
#     "PM2P5": [[0.0, 50.0, 0.0, 12.0],      # FIX: Was 9.0,
#              [51.0, 100.0, 12.1, 35.4],   # FIX: Was 9.1
#              [101.0, 150.0, 35.5, 55.4],
#              [151.0, 200.0, 55.5, 125.4],
#              [201.0, 300.0, 125.5, 225.4],
#              [301.0, 500.0, 225.5, 500.4]]
# }

# UAE EAQI (Emirati Air Quality Index) Breakpoints - Official 2023
# Source: https://airquality.ncm.gov.ae/resources/pdf/aqi-quickguide-en-2023.pdf
# Units: µg/m³ for all pollutants except CO (mg/m³)
# Each inner list: [low_aqi, high_aqi, low_breakpoint, high_breakpoint]

_aqi_break = {
    "CO": [[0.0, 50.0, 0.0, 5.4],      # Good (8-hour, mg/m³)
           [51.0, 100.0, 5.5, 10.4],    # Moderate
           [101.0, 150.0, 10.5, 14.4],  # Unhealthy for Sensitive Groups
           [151.0, 200.0, 14.5, 17.9],  # Unhealthy
           [201.0, 300.0, 18.0, 35.4],  # Very Unhealthy
           [301.0, 500.0, 35.5, 58.4]], # Hazardous
    
    "SO2": [[0.0, 50.0, 0.0, 92],       # Good (1-hour, µg/m³)
            [51.0, 100.0, 93, 350],     # Moderate
            [101.0, 150.0, 351, 485],   # Unhealthy for Sensitive Groups
            [151.0, 200.0, 486, 797],   # Unhealthy
            [201.0, 300.0, 798, 1583],  # Very Unhealthy
            [301.0, 500.0, 1584, 2631]],# Hazardous
    
    "NO2": [[0.0, 50.0, 0.0, 100],      # Good (1-hour, µg/m³)
            [51.0, 100.0, 101, 400],    # Moderate
            [101.0, 150.0, 401, 677],   # Unhealthy for Sensitive Groups
            [151.0, 200.0, 678, 1221],  # Unhealthy
            [201.0, 300.0, 1222, 2349], # Very Unhealthy
            [301.0, 500.0, 2350, 3853]],# Hazardous
    
    "O3": [[0.0, 50.0, 0.0, 100],       # Good (8-hour, µg/m³)
           [51.0, 100.0, 101, 120],     # Moderate
           [101.0, 150.0, 121, 167],    # Unhealthy for Sensitive Groups
           [151.0, 200.0, 168, 206],    # Unhealthy
           [201.0, 300.0, 207, 392],    # Very Unhealthy
           [301.0, 500.0, 793, 1184]],  # Hazardous
    
    "PM10": [[0.0, 50.0, 0.0, 75],      # Good (24-hour, µg/m³)
             [51.0, 100.0, 76, 150],    # Moderate
             [101.0, 150.0, 151, 250],  # Unhealthy for Sensitive Groups
             [151.0, 200.0, 251, 350],  # Unhealthy
             [201.0, 300.0, 351, 420],  # Very Unhealthy
             [301.0, 500.0, 421, 600]], # Hazardous
    
    "PM2P5": [[0.0, 50.0, 0.0, 50.4],     # Good (24-hour, µg/m³)
              [51.0, 100.0, 50.5, 60.4],  # Moderate
              [101.0, 150.0, 60.5, 75.4], # Unhealthy for Sensitive Groups
              [151.0, 200.0, 75.5, 150.4],# Unhealthy
              [201.0, 300.0, 150.5, 250.4],# Very Unhealthy
              [301.0, 500.0, 250.5, 500.4]]# Hazardous
}

In [ ]:
def load_data(data_path, pollutant: str, region):
    """
    Loads, merges, and standardises pollutant data from multiple Zarr files.
    """

    # Find all Zarr files within the specified data_path.
    zarr_files = list(data_path.glob('*.zarr'))
    # Filter the list of Zarr files to include only those containing the pollutant name.
    pollutant_zarrs = list(filter(lambda x: pollutant in str(x), zarr_files))
    # Initialize an empty list to store xarray DataArrays loaded from the Zarr files.
    da_list = []
    # Define the target Coordinate Reference System (CRS).
    dst_crs = "EPSG:3857"
    
    # Iterate through each pollutant-specific Zarr file.
    for zarr in pollutant_zarrs:
        print(f"Loading: {zarr}")
        # Open the Zarr file as an xarray Dataset, enabling automatic chunking for Dask parallelization.
        da = xr.open_zarr(zarr, chunks='auto')
        
        # Drop specific coordinates if the data source is 'CAMS'.
        if 'CAMS' in str(zarr):
            da = da.drop_vars(['number', 'step', 'surface', 'valid_time'], errors='ignore')
        
        # FIX: Drop band and spatial_ref to avoid coordinate mismatch during concatenation
        da = da.drop_vars(['band', 'spatial_ref'], errors='ignore')
        
        # Rename the first data variable in the Dataset to the pollutant's name.
        da = da.rename({f'{list(da.keys())[0]}': f'{pollutant}'})
        
        # Write the target CRS to the DataArray using odc-geo.
        da = da.odc.assign_crs(dst_crs)
        
        # Add diagnostic - check RAW values before filtering
        raw_min = float(da[pollutant].min().values)
        raw_max = float(da[pollutant].max().values)
        print(f"  - RAW data range: {raw_min:.2f} to {raw_max:.2f}")
        print(f"  - Time range: {da.time.min().values} to {da.time.max().values}")
        
        # Check if data contains -9999 (nodata marker from normalization)
        if raw_min == -9999.0 and raw_max == -9999.0:
            print(f"  - WARNING: File contains only -9999 values (corrupted)")
        
        # Filter out negative and zero values (invalid for pollutant concentrations)
        # This also converts -9999 to NaN
        da = da.where(da[pollutant] > 0)
        
        valid_count = int((da[pollutant].values > 0).sum())
        print(f"  - Valid positive values: {valid_count}")
        
        # Append the processed DataArray to the list, dropping duplicate time steps.
        da_list.append(da.drop_duplicates(dim=...))

    # Concatenate all DataArrays in the list along the 'time' dimension and sort by time.
    full_da = xr.concat(da_list, dim='time', join='override').sortby('time')
    # Select time steps where the time coordinate is not null.
    full_da = full_da.sel(time=full_da.time.notnull())
    # Remove duplicate time steps after concatenation and sorting.
    full_da = full_da.sel(time=~full_da.get_index("time").duplicated())

    # Return the merged, sorted, and CRS-defined DataArray.
    return full_da

In [4]:
# def load_data(data_path, pollutant: str, region):
#     """
#     Loads, merges, and standardises pollutant data from multiple Zarr files.

#     This function finds all Zarr files for a given pollutant in a directory,
#     loads them, drops unnecessary coordinates from different data sources (like
#     CAMS), and renames the primary data variable to the pollutant's name. It
#     then combines them into a single, time-sorted xarray.DataArray.

#     Args:
#         data_path (pathlib.Path): The path to the directory containing the input Zarr files.
#         pollutant (str): The name of the pollutant to load (e.g., 'SO2').
#         region (geopandas.GeoDataFrame): A GeoDataFrame defining the area of interest.
#                                          Note: This argument is not used in the function body.

#     Returns:
#         xarray.DataArray: A single DataArray containing the merged and standardised
#                           data for the specified pollutant.
#     """

#     # Find all Zarr files within the specified data_path.
#     zarr_files = list(data_path.glob('*.zarr'))
#     # Filter the list of Zarr files to include only those containing the pollutant name.
#     pollutant_zarrs = list(filter(lambda x: pollutant in str(x), zarr_files))
#     # Initialize an empty list to store xarray DataArrays loaded from the Zarr files.
#     da_list = []
#     # Iterate through each pollutant-specific Zarr file.
#     for zarr in pollutant_zarrs:
#         print(zarr)
#         # Open the Zarr file as an xarray Dataset, enabling automatic chunking for Dask parallelization.
#         da = xr.open_zarr(zarr, chunks='auto')
#         # Drop the 'spatial_ref' coordinate if the data source is not 'MOD'.
#         # if 'MOD' not in str(zarr):
#         #     da = da.drop_vars(['spatial_ref'])
#         # Drop specific coordinates ('number', 'step', 'surface', 'valid_time') if the data source is 'CAMS'.
#         if 'CAMS' in str(zarr):
#             da = da.drop_vars(['number', 'step', 'surface', 'valid_time'])
#         # Rename the first data variable in the Dataset to the pollutant's name.
#         da = da.rename({f'{list(da.keys())[0]}': f'{pollutant}'})
#         da = da.where(da[pollutant] > 0)
#         # Append the processed DataArray to the list, dropping duplicate time steps.
#         da_list.append(da.drop_duplicates(dim=...))

#     # Concatenate all DataArrays in the list along the 'time' dimension and sort by time.
#     full_da = xr.concat(da_list, dim='time').sortby('time')
#     # Select time steps where the time coordinate is not null.
#     full_da = full_da.sel(time=full_da.time.notnull())
#     # Remove duplicate time steps after concatenation and sorting.
#     full_da = full_da.sel(time=~full_da.get_index("time").duplicated())

#     # Define the target Coordinate Reference System (CRS).
#     dst_crs = "EPSG:3857"
#     # Write the target CRS to the DataArray using rioxarray.
#     full_da = full_da.rio.write_crs(dst_crs)

#     # Return the merged, sorted, and CRS-defined DataArray.
#     return full_da

In [5]:
def resample_data(data_array, start_date, end_date, sample_period, stat_method):
    """
    Resamples a DataArray to a specified time frequency using a given statistical method.

    Args:
        data_array (xarray.DataArray): The input time-series data to be resampled.
        start_date (str): The start date for the time slice (e.g., 'YYYY-MM-DD').
        end_date (str): The end date for the time slice (e.g., 'YYYY-MM-DD').
        sample_period (str): The target resampling frequency (e.g., '1D' for daily,
                             '1ME' for month-end, '1h' for hourly).
        stat_method (str): The statistical aggregation method to use ('mean',
                           'max', or 'median').

    Returns:
        xarray.DataArray: The temporally resampled DataArray.
    """

    # Slice the DataArray to the specified date range.
    da_sub = data_array.sel(time=slice(start_date, end_date))

    # Resample the data based on the specified sample_period and apply the chosen statistical method.
    if stat_method == 'mean':
        da_temporal = da_sub.resample(time=sample_period).mean()
    if stat_method == 'max':
        da_temporal = da_sub.resample(time=sample_period).max()
    if stat_method == 'median':
        da_temporal = da_sub.resample(time=sample_period).median()
    # Return the temporally resampled DataArray.
    return da_temporal

In [6]:
def calculate_compliance_days(data_array, pollutant, exceeds = False):
    """
    Creates a binary mask indicating compliance with WHO air quality guidelines.

    The function compares the data against predefined concentration limits for a
    given pollutant.

    Args:
        data_array (xarray.DataArray): DataArray of pollutant concentrations.
        pollutant (str): The name of the pollutant (e.g., 'SO2').
        exceeds (bool): If False (default), returns 1 for compliant values (below
                        the threshold) and 0 for non-compliant. If True, returns
                        1 for non-compliant values (exceeding the threshold) and 0 for compliant.

    Returns:
        xarray.DataArray: A DataArray with binary values (1 or 0) indicating compliance.
    """

    # Check if the 'exceeds' flag is set to True.
    if exceeds is True:
        # If 'exceeds' is True, return 1 where data_array is less than the threshold (compliant) and 0 otherwise (non-compliant).
        threshold_da = xr.where(data_array < _aq_complience_limits[f'{pollutant}_AQG'], 0, 1)
    else:
        # If 'exceeds' is False (default), return 1 where data_array is less than the threshold (compliant) and 0 otherwise (non-compliant).
        threshold_da = xr.where(data_array < _aq_complience_limits[f'{pollutant}_AQG'], 1, 0)
    # Return the binary DataArray indicating compliance.
    return threshold_da

In [7]:
def calculate_pollutant_aqi(data_array, pollutant):
    """
    Calculates the Air Quality Index (AQI) for a single pollutant.

    This function first converts concentration units for certain gases (O3, CO,
    NO2, SO2) to the appropriate scale for AQI calculation. It then applies the
    standard linear interpolation formula based on predefined concentration
    breakpoints.

    Args:
        data_array (xarray.Dataset): An xarray Dataset containing the pollutant
                                     as a data variable.
        pollutant (str): The name of the pollutant to calculate the AQI for
                         (e.g., 'PM2P5', 'O3').

    Returns:
        xarray.DataArray: A DataArray containing the calculated AQI values.
    """

    # Extract the specified pollutant's DataArray from the input Dataset.
    data_array = data_array[pollutant]
    # Convert units for specific gases to match the AQI breakpoint units (ug/m3 to ppm for O3, CO, NO2, SO2).
    if pollutant == 'O3':
        data_array = ((24.45 * data_array)/48)/1000 # conversion for O3
    if pollutant == 'CO':
        data_array = (24.45 * data_array)/28.01 # conversion for CO
    if pollutant == 'NO2':
        data_array = (24.45 * data_array)/46.01 # conversion for NO2
    if pollutant == 'SO2':
        data_array = (24.45 * data_array)/64.07 # conversion for SO2

    # Initialize aqi_thres with the input data_array. This will be updated iteratively.
    aqi_thres = data_array

    # Iterate through the AQI breakpoints for the given pollutant.
    for bp in _aqi_break[pollutant]:
        # Unpack the breakpoint values: low_aqi, high_aqi, low_breakpoint, high_breakpoint.
        low_aqi, high_aqi, low_bp, high_bp = bp
        # Apply the AQI linear interpolation formula for the current breakpoint range.
        # The formula is: AQI = [(I_high - I_low) / (C_high - C_low)] * (C - C_low) + I_low
        # where:
        # I_high = high_aqi (AQI value at the high breakpoint)
        # I_low = low_aqi (AQI value at the low breakpoint)
        # C_high = high_bp (pollutant concentration at the high breakpoint)
        # C_low = low_bp (pollutant concentration at the low breakpoint)
        # C = data_array (the pollutant concentration being evaluated)
        if low_aqi == 0.0:
             # For the first breakpoint (low_aqi = 0.0), apply the formula directly to the data_array.
             aqi_thres = xr.where((data_array >= low_bp) & (data_array <= high_bp), ((high_aqi - low_aqi) / (high_bp - low_bp)) * (data_array - low_bp) + low_aqi, data_array)
        else:
            # For subsequent breakpoints, apply the formula to the result of the previous interpolation step (aqi_thres).
            aqi_thres = xr.where((data_array >= low_bp) & (data_array <= high_bp), ((high_aqi - low_aqi) / (high_bp - low_bp)) * (data_array - low_bp) + low_aqi, aqi_thres)

    # Return the DataArray with calculated AQI values.
    return aqi_thres

In [8]:
def export_to_csv(data_array, output_path, pollutant, start_date, end_date, sample_period, stat_method, gdf):
    """
    Calculates zonal statistics for a set of regions and exports them to a CSV file.

    The function calculates a statistic (e.g., mean, max) for each polygon in the
    input GeoDataFrame from the raster data. The resulting table is pivoted to have
    regions as rows and time periods as columns and is saved as a CSV.

    Args:
        data_array (xarray.Dataset): The input dataset containing the pollutant data.
        output_path (pathlib.Path): The directory where the output CSV will be saved.
        pollutant (str): The name of the pollutant variable in the dataset.
        start_date (str): The start date of the analysis period.
        end_date (str): The end date of the analysis period.
        sample_period (str): The sampling period of the data (e.g., '1D', '1ME').
        stat_method (str): The statistical method used ('mean', 'max', 'median').
        gdf (geopandas.GeoDataFrame): A GeoDataFrame containing the regions for which
                                      to calculate zonal statistics.
    """

    # Ensure the 'OBJECTID' column in the GeoDataFrame is of integer type.
    gdf["OBJECTID"] = gdf.OBJECTID.astype(int)

    # Compute the DataArray to bring data from Dask to memory (if lazy loaded).
    data_array = data_array.compute()

    # Create a geocube from the GeoDataFrame with 'OBJECTID' as a measurement,
    # aligned with the grid of the input data_array.
    out_grid = make_geocube(
    vector_data=gdf,
    measurements=["OBJECTID"],
    like=data_array, # ensure the data are on the same grid
    )
    # Add the pollutant data from the input data_array to the geocube.
    out_grid[pollutant] = data_array[f'{pollutant}']

    # Group the geocube data by the 'OBJECTID' measurement and drop the spatial_ref coordinate.
    grouped_pollutant = out_grid.drop_vars("spatial_ref").groupby(out_grid.OBJECTID)
    # Calculate the specified statistical method for each group (region).
    if stat_method == 'mean':
        grid_stat = grouped_pollutant.mean()
    if stat_method == 'max':
        grid_stat = grouped_pollutant.max()
    if stat_method == 'median':
        grid_stat = grouped_pollutant.median()

    # Convert the resulting statistical grid to a pandas DataFrame.
    zonal_stats = grid_stat.to_dataframe()

    # Reset the index of the DataFrame to turn 'OBJECTID' and 'time' into columns.
    zonal_stats = zonal_stats.reset_index()

    # Create a list of formatted date strings based on the sample_period for column naming.
    dates = []
    if sample_period == '1YE':
        dates = [f'Y{d.year}' for d in list(zonal_stats['time'])]
    if sample_period == '1QE':
        dates = [f'Y{d.year}_Q{d.quarter}' for d in list(zonal_stats['time'])]
    if sample_period == '1ME':
        dates = [f'Y{d.year}_M{d.month}' for d in list(zonal_stats['time'])]
    if sample_period == '1D':
        dates = [f'Y{d.year}_M{d.month}_D{d.day}' for d in list(zonal_stats['time'])]
    if sample_period == '8H':
        dates = list(zonal_stats['time']) # For hourly, keep original datetime objects for now
    if sample_period == '1H':
        dates = list(zonal_stats['time']) # For hourly, keep original datetime objects for now

    # Replace the 'time' column in the zonal_stats DataFrame with the formatted date strings.
    for i, d in enumerate(dates):
        zonal_stats.loc[zonal_stats.index[i],'time']=d

    # Pivot the DataFrame to have 'OBJECTID' as the index, 'time' (formatted dates) as columns, and pollutant values as data.
    export_df = zonal_stats.pivot(index='OBJECTID', columns='time', values=pollutant)
    # Merge the pivoted statistics DataFrame with the original GeoDataFrame using 'OBJECTID'.
    export_gdf = gdf.merge(export_df, on='OBJECTID')

    # Define the columns to include in the final exported DataFrame (REGIONNAME and the formatted date columns).
    if sample_period == '1D':
        filter_cols = ['REGIONNAME'] + sorted(set(dates), key=lambda x: (int(x.split('_')[1][1:]), int(x.split('_')[2][1:])))
    elif sample_period == '1ME':
        filter_cols = ['REGIONNAME'] + sorted(set(dates), key=lambda x: (int(x.split('_')[1][1:])))
    elif sample_period == '1QE' or sample_period == '1YE':
        filter_cols = ['REGIONNAME'] + sorted(set(dates))
    else:
        filter_cols = ['REGIONNAME'] + dates

    # Select the desired columns for export.
    df_to_export = export_gdf[filter_cols]
    df_to_export = df_to_export.loc[:,~df_to_export.columns.duplicated()].copy()
    # Export the DataFrame to a CSV file with a descriptive filename based on pollutant, period, method, and date range.
    df_to_export.to_csv(str(output_path.joinpath(f'{pollutant}_{sample_period}_{stat_method}_{start_date}-{end_date}_Region_Stats.csv')))

In [ ]:
def export_data_tif(data_array, output_path, ids, pollutant, sample_period, stat_method):
    """
    Exports each time slice of a DataArray to a separate GeoTIFF file.

    It iterates through a list of time indices, creates a descriptive filename
    for each time step, and saves it as a GeoTIFF, avoiding overwriting existing files.

    Args:
        data_array (xarray.Dataset): The input dataset containing the pollutant data.
        output_path (pathlib.Path): The directory where the output GeoTIFFs will be saved.
        ids (list): A list of integer indices for the time dimension to be exported.
        pollutant (str): The name of the pollutant being processed.
        sample_period (str): The sampling period of the data, used for file naming.
        stat_method (str): The statistical method used, used for file naming.
    """

    # Iterate through the provided list of time indices.
    for i in ids:
        # Get the timestamp for the current time index.
        date = pd.Timestamp(data_array.isel(time=i).time.values)
        
        # Select the data slice
        da_slice = data_array[pollutant].isel(time=i)
        
        # Replace any NaN or inf with -9999
        da_slice = xr.where(np.isnan(da_slice) | np.isinf(da_slice), -9999, da_slice)
        
        # Ensure data is float32 to avoid precision issues
        da_slice = da_slice.astype('float32')
        
        # Set -9999 as the nodata value
        da_slice = da_slice.rio.write_nodata(-9999, inplace=True)
        
        # Generate filename and save based on the sample period
        if sample_period == '1YE':
            da_slice.rio.to_raster(output_path.joinpath(f'{pollutant}_{stat_method}_Y{date.year}.tif'), nodata=-9999, dtype='float32')
        elif sample_period == '1QE':
            da_slice.rio.to_raster(output_path.joinpath(f'{pollutant}_{stat_method}_Y{date.year}_Q{date.quarter}.tif'), nodata=-9999, dtype='float32')
        elif sample_period == '1ME':
            da_slice.rio.to_raster(output_path.joinpath(f'{pollutant}_{stat_method}_Y{date.year}_M{date.month}.tif'), nodata=-9999, dtype='float32')
        elif sample_period == '1D':
            da_slice.rio.to_raster(output_path.joinpath(f'{pollutant}_{stat_method}_Y{date.year}_M{date.month}_D{date.day}.tif'), nodata=-9999, dtype='float32')
        elif sample_period == '8H':
            da_slice.rio.to_raster(output_path.joinpath(f'{pollutant}_8H_{stat_method}_Y{date.year}_M{date.month}_D{date.day}_H{date.hour}.tif'), nodata=-9999, dtype='float32')
        elif sample_period == '1H':
            da_slice.rio.to_raster(output_path.joinpath(f'{pollutant}_1H_{stat_method}_Y{date.year}_M{date.month}_D{date.day}_H{date.hour}.tif'), nodata=-9999, dtype='float32')

In [ ]:
# def export_data_tif(data_array, output_path, ids, pollutant, sample_period, stat_method):
#     """
#     Exports each time slice of a DataArray to a separate GeoTIFF file.

#     It iterates through a list of time indices, creates a descriptive filename
#     for each time step, and saves it as a GeoTIFF, avoiding overwriting existing files.

#     Args:
#         data_array (xarray.Dataset): The input dataset containing the pollutant data.
#         output_path (pathlib.Path): The directory where the output GeoTIFFs will be saved.
#         ids (list): A list of integer indices for the time dimension to be exported.
#         pollutant (str): The name of the pollutant being processed.
#         sample_period (str): The sampling period of the data, used for file naming.
#         stat_method (str): The statistical method used, used for file naming.
#     """

#     # Iterate through the provided list of time indices.
#     for i in ids:
#         # Get the timestamp for the current time index.
#         date = pd.Timestamp(data_array.isel(time=i).time.values)
#         # Generate a filename based on the pollutant, statistical method, and the time period.
#         # Check if a file with the same name already exists to avoid overwriting.
#         if sample_period == '1YE':
#             # Filename for yearly data.
#             # Select the pollutant data for the current time slice, write nodata values, and save as GeoTIFF.
#             data_array[pollutant].isel(time=i).rio.write_nodata(np.nan, inplace=True).rio.to_raster(output_path.joinpath(f'{pollutant}_{stat_method}_Y{date.year}.tif'),nodata=-9999)
#         if sample_period == '1QE':
#             # Filename for quarterly data.
#             # Select the pollutant data for the current time slice, write nodata values, and save as GeoTIFF.
#             data_array[pollutant].isel(time=i).rio.write_nodata(np.nan, inplace=True).rio.to_raster(output_path.joinpath(f'{pollutant}_{stat_method}_Y{date.year}_Q{date.quarter}.tif'))
#         if sample_period == '1ME':
#             # Filename for monthly data.
#             # Select the pollutant data for the current time slice, write nodata values, and save as GeoTIFF.
#             data_array[pollutant].isel(time=i).rio.write_nodata(np.nan, inplace=True).rio.to_raster(output_path.joinpath(f'{pollutant}_{stat_method}_Y{date.year}_M{date.month}.tif'))
#         if sample_period == '1D':
#             # Filename for daily data.
#             # Select the pollutant data for the current time slice, write nodata values, and save as GeoTIFF.
#             data_array[pollutant].isel(time=i).rio.write_nodata(np.nan, inplace=True).rio.to_raster(output_path.joinpath(f'{pollutant}_{stat_method}_Y{date.year}_M{date.month}_D{date.day}.tif'))
#         if sample_period == '8H':
#             # Filename for 8-hourly data.
#             # Select the pollutant data for the current time slice, write nodata values, and save as GeoTIFF.
#             data_array[pollutant].isel(time=i).rio.write_nodata(np.nan, inplace=True).rio.to_raster(output_path.joinpath(f'{pollutant}_8H_{stat_method}_Y{date.year}_M{date.month}_D{date.day}_H{date.hour}.tif'))
#         if sample_period == '1H':
#             # Filename for 1-hourly data.
#             # Select the pollutant data for the current time slice, write nodata values, and save as GeoTIFF.
#             data_array[pollutant].isel(time=i).rio.write_nodata(np.nan, inplace=True).rio.to_raster(output_path.joinpath(f'{pollutant}_1H_{stat_method}_Y{date.year}_M{date.month}_D{date.day}_H{date.hour}.tif'))